# DS603 — Laboratory 09
## Sorting, Graph Traversal & Dynamic Programming

## Problem 01 — Sorting Instrumentation Engine

### Design Cell

Each function copies the input (`a = arr[:]`), so the caller's list is never mutated. Counting follows the lab convention: every evaluation of the key comparison is counted, including the one that ends the inner loop; the bound check `j >= 0` is not counted.

**Insertion Sort.** Invariant: at the start of iteration `i`, `a[0..i-1]` is sorted and holds the original first `i` elements. The key is compared against `a[j]` moving left; each `a[j] > key` causes one shift. Moves = shifts.
- Best case (sorted): $n-1$ comparisons, 0 shifts → $\Theta(n)$. Worst case (reversed): $n(n-1)/2$ comparisons and shifts → $\Theta(n^2)$. Space $O(1)$ extra.

**Selection Sort.** Invariant: at the start of iteration `i`, `a[0..i-1]` holds the `i` smallest elements in final sorted order. Find the minimum of `a[i..n-1]`, swap it into position `i` (a self-swap is skipped and not counted). Moves = swaps.
- Always $n(n-1)/2$ comparisons, at most $n-1$ swaps → $\Theta(n^2)$ in every case. Space $O(1)$.

**Bubble Sort (with `swapped` flag).** Invariant: after the sweep ending at index `end`, `a[end..n-1]` holds the largest elements in final positions. A sweep with no swap means the array is sorted, so we stop. Moves = swaps.
- Best case (sorted): one sweep, $n-1$ comparisons → $\Theta(n)$. Worst case (reversed): $n(n-1)/2$ comparisons and swaps → $\Theta(n^2)$. Space $O(1)$.

**`compare(arr)`** runs the three sorts on the same input and returns their comparison counts.

In [1]:
def insertion_sort(arr: list[int]) -> tuple[list[int], int, int]:
    """Return (sorted copy, comparisons, shifts)."""
    a = arr[:]
    comparisons = 0
    shifts = 0
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if a[j] > key:
                a[j + 1] = a[j]
                shifts += 1
                j -= 1
            else:
                break
        a[j + 1] = key
    return a, comparisons, shifts


def selection_sort(arr: list[int]) -> tuple[list[int], int, int]:
    """Return (sorted copy, comparisons, swaps); self-swaps are skipped."""
    a = arr[:]
    n = len(a)
    comparisons = 0
    swaps = 0
    for i in range(n - 1):
        smallest = i
        for j in range(i + 1, n):
            comparisons += 1
            if a[j] < a[smallest]:
                smallest = j
        if smallest != i:
            a[i], a[smallest] = a[smallest], a[i]
            swaps += 1
    return a, comparisons, swaps


def bubble_sort(arr: list[int]) -> tuple[list[int], int, int]:
    """Return (sorted copy, comparisons, swaps); uses the early-exit flag."""
    a = arr[:]
    n = len(a)
    comparisons = 0
    swaps = 0
    for end in range(n - 1, 0, -1):
        swapped = False
        for i in range(end):
            comparisons += 1
            if a[i] > a[i + 1]:
                a[i], a[i + 1] = a[i + 1], a[i]
                swaps += 1
                swapped = True
        if not swapped:
            break
    return a, comparisons, swaps


def compare(arr: list[int]) -> tuple[int, int, int]:
    """Comparison counts of (insertion, selection, bubble) on the same input."""
    return (insertion_sort(arr)[1], selection_sort(arr)[1],
            bubble_sort(arr)[1])


def process_sorting(text: str) -> list[str]:
    """Execute the Problem 01 command stream and return the output lines."""
    lines = text.strip().splitlines()
    q = int(lines[0])
    out: list[str] = []
    for line in lines[1:q + 1]:
        parts = line.split()
        cmd, values = parts[0], [int(v) for v in parts[1:]]
        if cmd == "INSERTION":
            a, c, s = insertion_sort(values)
            out.append(f"Insertion Sort: {a} | Comparisons: {c} | Shifts: {s}")
        elif cmd == "SELECTION":
            a, c, s = selection_sort(values)
            out.append(f"Selection Sort: {a} | Comparisons: {c} | Swaps: {s}")
        elif cmd == "BUBBLE":
            a, c, s = bubble_sort(values)
            out.append(f"Bubble Sort: {a} | Comparisons: {c} | Swaps: {s}")
        elif cmd == "COMPARE":
            ins, sel, bub = compare(values)
            out.append(f"Compare -> Insertion: {ins} | Selection: {sel} "
                       f"| Bubble: {bub}")
    return out

In [2]:
sample1_in = """6
INSERTION 10 30 40 20
SELECTION 10 30 40 20
BUBBLE 10 30 40 20
INSERTION 1 2 3 4 5
INSERTION 5 4 3 2 1
COMPARE 5 4 3 2 1"""

sample1_out = """Insertion Sort: [10, 20, 30, 40] | Comparisons: 5 | Shifts: 2
Selection Sort: [10, 20, 30, 40] | Comparisons: 6 | Swaps: 2
Bubble Sort: [10, 20, 30, 40] | Comparisons: 6 | Swaps: 2
Insertion Sort: [1, 2, 3, 4, 5] | Comparisons: 4 | Shifts: 0
Insertion Sort: [1, 2, 3, 4, 5] | Comparisons: 10 | Shifts: 10
Compare -> Insertion: 10 | Selection: 10 | Bubble: 10"""

result = process_sorting(sample1_in)
print("\n".join(result))
assert result == sample1_out.splitlines()

Insertion Sort: [10, 20, 30, 40] | Comparisons: 5 | Shifts: 2
Selection Sort: [10, 20, 30, 40] | Comparisons: 6 | Swaps: 2
Bubble Sort: [10, 20, 30, 40] | Comparisons: 6 | Swaps: 2
Insertion Sort: [1, 2, 3, 4, 5] | Comparisons: 4 | Shifts: 0
Insertion Sort: [1, 2, 3, 4, 5] | Comparisons: 10 | Shifts: 10
Compare -> Insertion: 10 | Selection: 10 | Bubble: 10


### Verification Cell

In [3]:
import random

sorts = [insertion_sort, selection_sort, bubble_sort]

# Empty and single-element arrays
for f in sorts:
    assert f([]) == ([], 0, 0)
    assert f([7]) == ([7], 0, 0)

# Already-sorted and reverse-sorted input (exact best/worst counts)
n = 10
asc, desc = list(range(n)), list(range(n, 0, -1))
assert insertion_sort(asc)[1:] == (n - 1, 0)
assert insertion_sort(desc)[1:] == (n * (n - 1) // 2, n * (n - 1) // 2)
assert selection_sort(asc)[1:] == (n * (n - 1) // 2, 0)
assert bubble_sort(asc)[1:] == (n - 1, 0)
assert bubble_sort(desc)[1:] == (n * (n - 1) // 2, n * (n - 1) // 2)

# Duplicates and random arrays against the sorted() oracle; caller not mutated
random.seed(1)
for _ in range(300):
    arr = [random.randint(-5, 5) for _ in range(random.randint(0, 30))]
    snapshot = arr[:]
    for f in sorts:
        assert f(arr)[0] == sorted(arr)
    assert arr == snapshot
print("Problem 01: all verification tests passed.")

Problem 01: all verification tests passed.


## Problem 02 — Divide-and-Conquer Sorting Lab

### Design Cell

**Quicksort (Lomuto, pivot = last element).** `partition(l, r)` keeps the invariant: at the start of each loop iteration `a[l..po-1] < pivot`, `a[po..i-1] >= pivot`, `a[r] = pivot`. After the loop, swapping `a[po]` and `a[r]` places the pivot in its final position. Every evaluation of `a[i] < pivot` is counted. Recursion depth is recorded only for calls with `l < r`, the first call being depth 1. The function works on a copy of the input.
- Time: best/average $T(n)=2T(n/2)+\Theta(n)=\Theta(n\log n)$; worst (sorted/reversed input) $T(n)=T(n-1)+\Theta(n)=\Theta(n^2)$ with depth $n-1$.
- Space: in place; recursion stack $O(\log n)$ average, $O(n)$ worst.

**Merge Sort (top-down, `mid = len // 2`).** Merge invariant: `out` holds the smallest `i + j` elements of the two halves in sorted order. Each loop iteration of the merge performs one key comparison and is counted. On a tie the element from the left half is taken (`right < left` is strict), so the sort is stable.
- Time: $T(n)=2T(n/2)+\Theta(n)=\Theta(n\log n)$ in every case ($\log_2 n$ levels × $\Theta(n)$ work per level).
- Space: $\Theta(n)$ extra for the merge buffers; not in place.

**`stable_sort(records)`** uses the same merge sort, comparing only the key of each `(key, label)` pair, so records with equal keys keep their original order. Same complexity as Merge Sort.

In [4]:
from typing import Callable, TypeVar

T = TypeVar("T")


def quick_sort(arr: list[int]) -> tuple[list[int], int, int]:
    """Return (sorted copy, comparisons, max recursion depth)."""
    a = arr[:]
    comparisons = 0
    max_depth = 0

    def partition(l: int, r: int) -> int:
        nonlocal comparisons
        pivot = a[r]
        po = l
        for i in range(l, r):
            comparisons += 1
            if a[i] < pivot:
                a[i], a[po] = a[po], a[i]
                po += 1
        a[po], a[r] = a[r], a[po]
        return po

    def sort(l: int, r: int, depth: int) -> None:
        nonlocal max_depth
        if l < r:
            max_depth = max(max_depth, depth)
            q = partition(l, r)
            sort(l, q - 1, depth + 1)
            sort(q + 1, r, depth + 1)

    sort(0, len(a) - 1, 1)
    return a, comparisons, max_depth


def _merge_sort_by(s: list[T], key: Callable[[T], int],
                   counter: list[int]) -> list[T]:
    """Top-down merge sort on key(x); counter[0] accumulates comparisons."""
    if len(s) <= 1:
        return s[:]
    mid = len(s) // 2
    left = _merge_sort_by(s[:mid], key, counter)
    right = _merge_sort_by(s[mid:], key, counter)
    out: list[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        counter[0] += 1
        if key(right[j]) < key(left[i]):   # strict: ties come from the left
            out.append(right[j])
            j += 1
        else:
            out.append(left[i])
            i += 1
    out.extend(left[i:])
    out.extend(right[j:])
    return out


def merge_sort(arr: list[int]) -> tuple[list[int], int]:
    """Return (sorted copy, comparisons made in the merge steps)."""
    counter = [0]
    result = _merge_sort_by(arr, lambda x: x, counter)
    return result, counter[0]


def stable_sort(records: list[tuple[int, str]]
                ) -> tuple[list[tuple[int, str]], int]:
    """Stable merge sort of (key, label) pairs by key only."""
    counter = [0]
    result = _merge_sort_by(records, lambda r: r[0], counter)
    return result, counter[0]


def process_dc_sorting(text: str) -> list[str]:
    """Execute the Problem 02 command stream and return the output lines."""
    lines = text.strip().splitlines()
    q = int(lines[0])
    out: list[str] = []
    for line in lines[1:q + 1]:
        parts = line.split()
        cmd = parts[0]
        if cmd == "QUICK":
            a, c, d = quick_sort([int(v) for v in parts[1:]])
            out.append(f"Quick Sort: {a} | Comparisons: {c} | Max Depth: {d}")
        elif cmd == "MERGE":
            a, c = merge_sort([int(v) for v in parts[1:]])
            out.append(f"Merge Sort: {a} | Comparisons: {c}")
        elif cmd == "STABLE":
            recs = []
            for tok in parts[1:]:
                k, lab = tok.split(":")
                recs.append((int(k), lab))
            res, c = stable_sort(recs)
            body = ", ".join(f"{k}:{lab}" for k, lab in res)
            out.append(f"Stable Sort: [{body}] | Comparisons: {c}")
    return out

In [5]:
sample2_in = """5
QUICK 5 2 9 4 3 7 6 1
QUICK 1 2 3 4 5
MERGE 10 50 7 62 17 41 88 3 25 39
MERGE 17 22 30 4 33 18 66 10
STABLE 3:a 1:b 3:c 2:d 1:e"""

sample2_out = """Quick Sort: [1, 2, 3, 4, 5, 6, 7, 9] | Comparisons: 18 | Max Depth: 4
Quick Sort: [1, 2, 3, 4, 5] | Comparisons: 10 | Max Depth: 4
Merge Sort: [3, 7, 10, 17, 25, 39, 41, 50, 62, 88] | Comparisons: 22
Merge Sort: [4, 10, 17, 18, 22, 30, 33, 66] | Comparisons: 16
Stable Sort: [1:b, 1:e, 2:d, 3:a, 3:c] | Comparisons: 8"""

result = process_dc_sorting(sample2_in)
print("\n".join(result))
assert result == sample2_out.splitlines()

Quick Sort: [1, 2, 3, 4, 5, 6, 7, 9] | Comparisons: 18 | Max Depth: 4
Quick Sort: [1, 2, 3, 4, 5] | Comparisons: 10 | Max Depth: 4
Merge Sort: [3, 7, 10, 17, 25, 39, 41, 50, 62, 88] | Comparisons: 22
Merge Sort: [4, 10, 17, 18, 22, 30, 33, 66] | Comparisons: 16
Stable Sort: [1:b, 1:e, 2:d, 3:a, 3:c] | Comparisons: 8


### Verification Cell

In [6]:
random.seed(2)

# Empty and single-element arrays
assert quick_sort([]) == ([], 0, 0) and quick_sort([4]) == ([4], 0, 0)
assert merge_sort([]) == ([], 0) and merge_sort([4]) == ([4], 0)

# Already-sorted and reverse-sorted input: quadratic quicksort worst case
n = 12
for arr in (list(range(n)), list(range(n, 0, -1))):
    s, c, d = quick_sort(arr)
    assert s == sorted(arr) and c == n * (n - 1) // 2 and d == n - 1
    assert merge_sort(arr)[0] == sorted(arr)

# Duplicates, random arrays, no mutation of the caller's list
for _ in range(300):
    arr = [random.randint(-5, 5) for _ in range(random.randint(0, 40))]
    snapshot = arr[:]
    assert quick_sort(arr)[0] == sorted(arr)
    assert merge_sort(arr)[0] == sorted(arr)
    assert arr == snapshot

# Stability check: equal keys keep their original order
for _ in range(300):
    recs = [(random.randint(0, 4), chr(97 + i % 26)) for i in range(20)]
    tagged = [(k, i) for i, (k, _) in enumerate(recs)]
    res_tagged, _ = stable_sort(tagged)
    assert res_tagged == sorted(tagged, key=lambda r: r[0])  # sorted() is stable
print("Problem 02: all verification tests passed.")

Problem 02: all verification tests passed.


## Problem 03 — Graph Traversal Engine

### Design Cell

**Representation.** Undirected simple graph on `0..n-1` stored as an adjacency list; each new neighbour is inserted into its list at the correct position, so every neighbour list is sorted ascending (no built-in sort). Space $\Theta(n+m)$.

**Traversal order rules.**
- `dfs(s)` (recursive): mark `v` visited, append it to the order, then recurse into each unvisited neighbour in ascending order.
- `dfs_iterative(s)`: an explicit stack of `(vertex, next-index)` frames. The top frame resumes scanning its neighbour list from `next-index`; on finding an unvisited neighbour `u` it pushes `(v, i+1)` back, marks and records `u`, and pushes `(u, 0)` (go deeper); if the list is exhausted the frame stays popped (backtrack). This reproduces the recursive order exactly.
- `bfs(s)`: FIFO queue (`collections.deque`); a vertex is marked (`dist` set) when it is **enqueued**, neighbours are scanned in ascending order. Invariant: the queue holds vertices at distance $d$ followed by vertices at distance $d+1$, so the first discovery of a vertex is along a fewest-edges path.

**Other operations.**
- `components()`: restart BFS from each still-unlabelled vertex in increasing order, labelling each vertex with a component id; then sweep `v = 0..n-1` appending `v` to its component. Each list is therefore sorted and components are ordered by smallest vertex.
- `shortest_path(s, t)`: BFS from `s`, follow `parent[]` from `t` back to `s`, reverse; `None` if `dist[t] == -1`.
- `has_cycle()`: DFS over every component; a visited neighbour `w` that is not the parent of `v` closes a cycle.

**Complexity.** Every traversal labels each vertex a constant number of times and examines each edge a constant number of times: $O(n+m)$ time, $O(n)$ extra space (recursion/stack depth, queue width, visited arrays). Building the graph with sorted insertion costs $O(\sum_v \deg(v)^2)$ in the worst case, which is negligible for $N \le 50$.

In [7]:
from collections import deque


class Graph:
    """Undirected simple graph with ascending-sorted adjacency lists."""

    def __init__(self, n: int, edges: list[tuple[int, int]]) -> None:
        self.n = n
        self.adj: list[list[int]] = [[] for _ in range(n)]
        for u, v in edges:
            self._insert_sorted(self.adj[u], v)
            self._insert_sorted(self.adj[v], u)
        self.dist: list[int] = [-1] * n
        self.parent: list[int] = [-1] * n

    @staticmethod
    def _insert_sorted(lst: list[int], x: int) -> None:
        """Insert x into the ascending list lst, keeping it ascending."""
        lst.append(x)
        i = len(lst) - 1
        while i > 0 and lst[i - 1] > x:
            lst[i] = lst[i - 1]
            i -= 1
        lst[i] = x

    def dfs(self, s: int) -> list[int]:
        """Recursive DFS visit order from s."""
        visited = [False] * self.n
        order: list[int] = []

        def visit(v: int) -> None:
            visited[v] = True
            order.append(v)
            for u in self.adj[v]:
                if not visited[u]:
                    visit(u)

        visit(s)
        return order

    def dfs_iterative(self, s: int) -> list[int]:
        """Explicit-stack DFS with (vertex, next-index) frames."""
        visited = [False] * self.n
        visited[s] = True
        order = [s]
        stack: list[tuple[int, int]] = [(s, 0)]
        while stack:
            v, i = stack.pop()
            while i < len(self.adj[v]) and visited[self.adj[v][i]]:
                i += 1
            if i == len(self.adj[v]):
                continue
            u = self.adj[v][i]
            stack.append((v, i + 1))
            visited[u] = True
            order.append(u)
            stack.append((u, 0))
        return order

    def bfs(self, s: int) -> list[int]:
        """BFS visit order from s; fills self.dist and self.parent."""
        self.dist = [-1] * self.n
        self.parent = [-1] * self.n
        self.dist[s] = 0
        q = deque([s])
        order: list[int] = []
        while q:
            v = q.popleft()
            order.append(v)
            for u in self.adj[v]:
                if self.dist[u] == -1:
                    self.dist[u] = self.dist[v] + 1
                    self.parent[u] = v
                    q.append(u)
        return order

    def components(self) -> list[list[int]]:
        """Connected components, each sorted, ordered by smallest vertex."""
        comp_id = [-1] * self.n
        count = 0
        for start in range(self.n):
            if comp_id[start] != -1:
                continue
            comp_id[start] = count
            q = deque([start])
            while q:
                v = q.popleft()
                for u in self.adj[v]:
                    if comp_id[u] == -1:
                        comp_id[u] = count
                        q.append(u)
            count += 1
        comps: list[list[int]] = [[] for _ in range(count)]
        for v in range(self.n):
            comps[comp_id[v]].append(v)
        return comps

    def shortest_path(self, s: int, t: int) -> list[int] | None:
        """Fewest-edges path from s to t, or None if unreachable."""
        self.bfs(s)
        if self.dist[t] == -1:
            return None
        path: list[int] = []
        v = t
        while v != -1:
            path.append(v)
            v = self.parent[v]
        return path[::-1]

    def has_cycle(self) -> bool:
        """DFS-based cycle detection over all components."""
        visited = [False] * self.n

        def visit(v: int, parent: int) -> bool:
            visited[v] = True
            for w in self.adj[v]:
                if not visited[w]:
                    if visit(w, v):
                        return True
                elif w != parent:
                    return True
            return False

        return any(not visited[v] and visit(v, -1) for v in range(self.n))


def process_graph(text: str) -> list[str]:
    """Execute the Problem 03 input and return the output lines."""
    lines = text.strip().splitlines()
    n, m = map(int, lines[0].split())
    edges = [tuple(map(int, lines[1 + i].split())) for i in range(m)]
    g = Graph(n, edges)
    q = int(lines[1 + m])
    out: list[str] = []
    for line in lines[2 + m:2 + m + q]:
        parts = line.split()
        cmd = parts[0]
        if cmd == "DFS":
            out.append(f"DFS: {g.dfs(int(parts[1]))}")
        elif cmd == "DFSITER":
            out.append(f"DFS (iterative): {g.dfs_iterative(int(parts[1]))}")
        elif cmd == "BFS":
            out.append(f"BFS: {g.bfs(int(parts[1]))}")
        elif cmd == "COMPONENTS":
            comps = g.components()
            out.append(f"Components ({len(comps)}): {comps}")
        elif cmd == "PATH":
            s, t = int(parts[1]), int(parts[2])
            path = g.shortest_path(s, t)
            if path is None:
                out.append(f"Shortest Path {s}->{t}: No Path")
            else:
                out.append(f"Shortest Path {s}->{t}: {path} "
                           f"({len(path) - 1} edges)")
        elif cmd == "CYCLE":
            out.append(f"Cycle: {g.has_cycle()}")
    return out

In [8]:
sample3_in = """8 6
0 1
0 2
1 3
2 3
3 4
5 6
7
DFS 0
DFSITER 0
BFS 0
COMPONENTS
PATH 0 4
PATH 0 6
CYCLE"""

sample3_out = """DFS: [0, 1, 3, 2, 4]
DFS (iterative): [0, 1, 3, 2, 4]
BFS: [0, 1, 2, 3, 4]
Components (3): [[0, 1, 2, 3, 4], [5, 6], [7]]
Shortest Path 0->4: [0, 1, 3, 4] (3 edges)
Shortest Path 0->6: No Path
Cycle: True"""

result = process_graph(sample3_in)
print("\n".join(result))
assert result == sample3_out.splitlines()

DFS: [0, 1, 3, 2, 4]
DFS (iterative): [0, 1, 3, 2, 4]
BFS: [0, 1, 2, 3, 4]
Components (3): [[0, 1, 2, 3, 4], [5, 6], [7]]
Shortest Path 0->4: [0, 1, 3, 4] (3 edges)
Shortest Path 0->6: No Path
Cycle: True


### Verification Cell

In [9]:
# Isolated vertices / graph with no edges
g = Graph(4, [])
assert g.components() == [[0], [1], [2], [3]]
assert g.dfs(2) == [2] and g.dfs_iterative(2) == [2] and g.bfs(2) == [2]
assert g.shortest_path(0, 3) is None and g.shortest_path(1, 1) == [1]
assert g.has_cycle() is False

# Single vertex
assert Graph(1, []).components() == [[0]]

# Disconnected forest (no cycle) vs. adding one edge (cycle)
forest = [(0, 1), (1, 2), (3, 4)]
assert Graph(5, forest).has_cycle() is False
assert Graph(5, forest + [(0, 2)]).has_cycle() is True

# Recursive and iterative DFS agree; BFS distances equal path lengths
random.seed(3)
for _ in range(200):
    n = random.randint(1, 15)
    pairs = [(u, v) for u in range(n) for v in range(u + 1, n)]
    edges = random.sample(pairs, random.randint(0, len(pairs)))
    g = Graph(n, edges)
    for s in range(n):
        assert g.dfs(s) == g.dfs_iterative(s)
        g.bfs(s)
        for t in range(n):
            p = g.shortest_path(s, t)
            assert (p is None) == (g.dist[t] == -1)
            if p is not None:
                assert len(p) - 1 == g.dist[t] and p[0] == s and p[-1] == t
    # union of components is every vertex exactly once
    assert sorted(v for c in g.components() for v in c) == list(range(n))
    # simple graph has a cycle iff m > n - (#components)
    assert g.has_cycle() == (len(edges) > n - len(g.components()))
print("Problem 03: all verification tests passed.")

Problem 03: all verification tests passed.


## Problem 04 — Dynamic Programming Engine (Matrix-Chain & LCS)

### Design Cell

**Matrix-Chain Multiplication.**
- *State:* $N_{i,j}$ = minimum scalar multiplications to compute $A_i \cdots A_j$, where $A_i$ is $d_{i-1}\times d_i$.
- *Recurrence:* $N_{i,j} = \min_{i\le k<j}\{N_{i,k} + N_{k+1,j} + d_{i-1}d_kd_j\}$; the winning $k$ is stored in $K[i][j]$ (strict `<`, so the smallest $k$ wins ties).
- *Base cases:* $N_{i,i} = 0$.
- *Evaluation order:* increasing chain length $L = 2,\dots,n$, so both sub-chains are already computed.
- *Answer:* $N_{1,n}$; the parenthesisation is rebuilt recursively from $K$ as `(left right)`, a single matrix printing as `Ai`.
- *Complexity:* $\Theta(n^2)$ states × up to $n-1$ splits → $\Theta(n^3)$ time; $\Theta(n^2)$ space for $N$ and $K$.

**Longest Common Subsequence.**
- *State:* $c[i][j]$ = length of the LCS of the prefixes $X_i$ and $Y_j$.
- *Recurrence:* $c[i][j] = c[i-1][j-1] + 1$ if $x_i = y_j$, otherwise $\max(c[i-1][j],\, c[i][j-1])$.
- *Base cases:* $c[0][j] = c[i][0] = 0$.
- *Evaluation order:* row by row, left to right (each cell needs its left, upper and upper-left neighbours).
- *Answer:* $c[m][n]$. *Reconstruction:* from $(m,n)$, on a match output the symbol and move diagonally; otherwise move up if $c[i-1][j] \ge c[i][j-1]$, else left; reverse the collected symbols.
- *Complexity:* `lcs` and `lcs_table` are $\Theta(mn)$ time and $\Theta(mn)$ space; back-tracking adds $O(m+n)$.

In [10]:
def matrix_chain(dims: list[int]) -> tuple[int, str]:
    """Return (minimum scalar multiplications, parenthesisation)."""
    n = len(dims) - 1
    N = [[0] * (n + 1) for _ in range(n + 1)]
    K = [[0] * (n + 1) for _ in range(n + 1)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            N[i][j] = float("inf")
            for k in range(i, j):
                cost = N[i][k] + N[k + 1][j] + dims[i - 1] * dims[k] * dims[j]
                if cost < N[i][j]:
                    N[i][j], K[i][j] = cost, k

    def paren(i: int, j: int) -> str:
        if i == j:
            return f"A{i}"
        k = K[i][j]
        return f"({paren(i, k)} {paren(k + 1, j)})"

    return N[1][n], paren(1, n)


def lcs_table(x: str, y: str) -> list[list[int]]:
    """Full (m+1) x (n+1) LCS length table."""
    m, n = len(x), len(y)
    c = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                c[i][j] = c[i - 1][j - 1] + 1
            else:
                c[i][j] = max(c[i - 1][j], c[i][j - 1])
    return c


def lcs(x: str, y: str) -> tuple[int, str]:
    """Return (LCS length, one LCS) using the 'move up first' tie rule."""
    c = lcs_table(x, y)
    i, j = len(x), len(y)
    out: list[str] = []
    while i > 0 and j > 0:
        if x[i - 1] == y[j - 1]:
            out.append(x[i - 1])
            i -= 1
            j -= 1
        elif c[i - 1][j] >= c[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return c[len(x)][len(y)], "".join(reversed(out))


def process_dp(text: str) -> list[str]:
    """Execute the Problem 04 command stream and return the output lines."""
    lines = text.strip().splitlines()
    q = int(lines[0])
    out: list[str] = []
    for line in lines[1:q + 1]:
        parts = line.split()
        cmd = parts[0]
        if cmd == "MCM":
            ops, order = matrix_chain([int(v) for v in parts[1:]])
            out.append(f"Min Ops: {ops} | Order: {order}")
        elif cmd == "LCS":
            length, seq = lcs(parts[1], parts[2])
            out.append(f"LCS Length: {length} | LCS: {seq}")
        elif cmd == "LCSTABLE":
            x, y = parts[1], parts[2]
            out.append(f"LCS Table (rows: {x}, cols: {y}):")
            for i, row in enumerate(lcs_table(x, y)):
                out.append(f"Row {i}: {row}")
    return out

In [11]:
sample4_in = """6
MCM 3 100 5 5
MCM 10 5 10 5 10
MCM 101 11 9 100 99
LCS ABCB BDCAB
LCSTABLE ABCB BDCAB
LCS AGGTAB GXTXAYB"""

sample4_out = """Min Ops: 1575 | Order: ((A1 A2) A3)
Min Ops: 1000 | Order: (A1 ((A2 A3) A4))
Min Ops: 189090 | Order: ((A1 A2) (A3 A4))
LCS Length: 3 | LCS: BCB
LCS Table (rows: ABCB, cols: BDCAB):
Row 0: [0, 0, 0, 0, 0, 0]
Row 1: [0, 0, 0, 0, 1, 1]
Row 2: [0, 1, 1, 1, 1, 2]
Row 3: [0, 1, 1, 2, 2, 2]
Row 4: [0, 1, 1, 2, 2, 3]
LCS Length: 4 | LCS: GTAB"""

result = process_dp(sample4_in)
print("\n".join(result))
assert result == sample4_out.splitlines()

Min Ops: 1575 | Order: ((A1 A2) A3)
Min Ops: 1000 | Order: (A1 ((A2 A3) A4))
Min Ops: 189090 | Order: ((A1 A2) (A3 A4))
LCS Length: 3 | LCS: BCB
LCS Table (rows: ABCB, cols: BDCAB):
Row 0: [0, 0, 0, 0, 0, 0]
Row 1: [0, 0, 0, 0, 1, 1]
Row 2: [0, 1, 1, 1, 1, 2]
Row 3: [0, 1, 1, 2, 2, 2]
Row 4: [0, 1, 1, 2, 2, 3]
LCS Length: 4 | LCS: GTAB


### Verification Cell

In [12]:
from itertools import combinations

# Single-matrix chain and two-matrix chain
assert matrix_chain([7, 3]) == (0, "A1")
assert matrix_chain([2, 3, 4]) == (24, "(A1 A2)")

# Brute force over all parenthesisations for small random chains
def brute_mcm(d: list[int], i: int, j: int) -> int:
    if i == j:
        return 0
    return min(brute_mcm(d, i, k) + brute_mcm(d, k + 1, j)
               + d[i - 1] * d[k] * d[j] for k in range(i, j))

random.seed(4)
for _ in range(100):
    d = [random.randint(1, 30) for _ in range(random.randint(2, 7))]
    assert matrix_chain(d)[0] == brute_mcm(d, 1, len(d) - 1)

# Empty and identical strings
assert lcs("", "") == (0, "") and lcs("ABC", "") == (0, "")
assert lcs("ABCD", "ABCD") == (4, "ABCD")
assert lcs("ABC", "XYZ") == (0, "")
assert process_dp("1\nLCS ABC XYZ") == ["LCS Length: 0 | LCS: "]
assert lcs_table("", "AB") == [[0, 0, 0]]

# Brute-force LCS length and validity of the reconstructed subsequence
def is_subseq(s: str, t: str) -> bool:
    it = iter(t)
    return all(ch in it for ch in s)

for _ in range(200):
    x = "".join(random.choice("ABC") for _ in range(random.randint(0, 7)))
    y = "".join(random.choice("ABC") for _ in range(random.randint(0, 7)))
    best = max((r for r in range(len(x) + 1)
                for comb in combinations(x, r)
                if is_subseq("".join(comb), y)), default=0)
    length, seq = lcs(x, y)
    assert length == best == len(seq)
    assert is_subseq(seq, x) and is_subseq(seq, y)
print("Problem 04: all verification tests passed.")

Problem 04: all verification tests passed.


## Problem 05 — Cargo Loading Optimiser (Knapsack Engine)

### Design Cell

**0/1 Knapsack (DP, `solve_01`).**
- *State:* $B[k][w]$ = maximum benefit using only the first $k$ items with capacity $w$.
- *Recurrence:* $B[k][w] = B[k-1][w]$ if $w_k > w$; otherwise $B[k][w] = \max(B[k-1][w],\; B[k-1][w-w_k] + b_k)$.
- *Base cases:* $B[0][w] = 0$ for all $w$, and $B[k][0] = 0$.
- *Evaluation order:* row $k = 1..n$, each row left to right (row $k$ depends only on row $k-1$).
- *Answer cell:* $B[n][W]$.
- *Reconstruction:* for $k = n..1$, item $k$ was taken iff $B[k][w] \ne B[k-1][w]$; then $w \leftarrow w - w_k$. Ids are reported ascending.
- *Complexity:* $\Theta(nW)$ time and space. This is **pseudo-polynomial**: it is polynomial in the numeric value $W$, but $W$ is written with only $\log_2 W$ bits, so the running time is exponential in the size of the input encoding.

**Fractional Knapsack (greedy, `solve_fractional`).** Sort items by $b/w$ descending (ties by smaller id), take whole items while they fit, then the exact fraction $(W - \text{used})/w$ of the first item that does not fit, and stop. Optimal by the exchange argument (replacing weight of a lower-ratio item by a higher-ratio item never decreases benefit). $\Theta(n\log n)$ time for the sort, $O(n)$ space.

**Naive greedy 0/1 loader (`solve_greedy_01`).** Same ratio order, but take an item only if it fits entirely and continue scanning after a failure. $\Theta(n\log n)$ time. It is not optimal; the gap is reported as the 0/1 optimum minus the greedy value.

**Validation.** `ValueError` for non-positive (or non-integer) benefit/weight, negative capacity, or a solve request before the capacity is set. Ratios use `fractions.Fraction` so equal ratios tie exactly.

In [13]:
from fractions import Fraction


class CargoOptimizer:
    """0/1, fractional and naive-greedy knapsack solvers."""

    def __init__(self) -> None:
        self.items: list[tuple[int, int]] = []   # (benefit, weight)
        self.capacity: int | None = None

    def add_item(self, benefit: int, weight: int) -> int:
        """Append an item and return its 1-based id."""
        if (not isinstance(benefit, int) or not isinstance(weight, int)
                or benefit <= 0 or weight <= 0):
            raise ValueError("benefit and weight must be positive integers")
        self.items.append((benefit, weight))
        return len(self.items)

    def set_capacity(self, capacity: int) -> None:
        """Set the knapsack capacity (non-negative integer)."""
        if not isinstance(capacity, int) or capacity < 0:
            raise ValueError("capacity must be a non-negative integer")
        self.capacity = capacity

    def _require_capacity(self) -> int:
        if self.capacity is None:
            raise ValueError("capacity has not been set")
        return self.capacity

    def _ratio_order(self) -> list[int]:
        """0-based indices by benefit/weight descending, ties by smaller id."""
        return sorted(range(len(self.items)),
                      key=lambda i: (-Fraction(*self.items[i]), i))

    def solve_01(self) -> tuple[int, list[int], int]:
        """Exact 0/1 knapsack by bottom-up DP."""
        cap = self._require_capacity()
        n = len(self.items)
        B = [[0] * (cap + 1) for _ in range(n + 1)]
        for k in range(1, n + 1):
            b, w = self.items[k - 1]
            for x in range(cap + 1):
                B[k][x] = B[k - 1][x]
                if w <= x:
                    B[k][x] = max(B[k][x], B[k - 1][x - w] + b)
        chosen: list[int] = []
        x = cap
        for k in range(n, 0, -1):
            if B[k][x] != B[k - 1][x]:
                chosen.append(k)
                x -= self.items[k - 1][1]
        chosen.reverse()
        weight = sum(self.items[k - 1][1] for k in chosen)
        return B[n][cap], chosen, weight

    def solve_fractional(self) -> tuple[float, dict[int, float]]:
        """Greedy fractional knapsack; fractions in consideration order."""
        cap = self._require_capacity()
        total = 0.0
        used = 0
        fractions: dict[int, float] = {}
        for i in self._ratio_order():
            b, w = self.items[i]
            if used + w <= cap:
                fractions[i + 1] = 1.0
                used += w
                total += b
            else:
                frac = (cap - used) / w
                fractions[i + 1] = frac
                total += b * frac
                break
        return total, fractions

    def solve_greedy_01(self) -> tuple[int, list[int], int]:
        """Naive ratio-first 0/1 loader (keeps scanning after a misfit)."""
        cap = self._require_capacity()
        total = 0
        used = 0
        taken: list[int] = []
        for i in self._ratio_order():
            b, w = self.items[i]
            if used + w <= cap:
                taken.append(i + 1)
                used += w
                total += b
        return total, sorted(taken), used


def process_cargo(text: str) -> list[str]:
    """Execute the Problem 05 command stream and return the output lines."""
    lines = text.strip().splitlines()
    q = int(lines[0])
    opt = CargoOptimizer()
    out: list[str] = []
    for line in lines[1:q + 1]:
        parts = line.split()
        cmd = parts[0]
        try:
            if cmd == "ADD":
                b, w = int(parts[1]), int(parts[2])
                item_id = opt.add_item(b, w)
                out.append(f"Added: item {item_id} (benefit={b}, weight={w})")
            elif cmd == "CAP":
                opt.set_capacity(int(parts[1]))
                out.append(f"Capacity set: {opt.capacity}")
            elif cmd == "ZEROONE":
                best, ids, wt = opt.solve_01()
                out.append(f"0/1 Optimal: {best} | Items: {ids} | Weight: {wt}")
            elif cmd == "FRACTIONAL":
                best, fr = opt.solve_fractional()
                body = ", ".join(f"{k}: {v:.2f}" for k, v in fr.items())
                out.append(f"Fractional Optimal: {best:.2f} | "
                           f"Fractions: {{{body}}}")
            elif cmd == "GREEDY":
                g_best, ids, wt = opt.solve_greedy_01()
                gap = opt.solve_01()[0] - g_best
                out.append(f"Greedy 0/1 (by ratio): {g_best} | Items: {ids} "
                           f"| Weight: {wt} | Gap vs Optimal: {gap}")
        except ValueError as exc:
            out.append(f"Error: {exc}")
    return out

In [14]:
sample5_in = """9
ADD 1 1
ADD 6 2
ADD 18 5
ADD 22 6
ADD 28 7
CAP 11
ZEROONE
FRACTIONAL
GREEDY"""

sample5_out = """Added: item 1 (benefit=1, weight=1)
Added: item 2 (benefit=6, weight=2)
Added: item 3 (benefit=18, weight=5)
Added: item 4 (benefit=22, weight=6)
Added: item 5 (benefit=28, weight=7)
Capacity set: 11
0/1 Optimal: 40 | Items: [3, 4] | Weight: 11
Fractional Optimal: 42.67 | Fractions: {5: 1.00, 4: 0.67}
Greedy 0/1 (by ratio): 35 | Items: [1, 2, 5] | Weight: 10 | Gap vs Optimal: 5"""

result = process_cargo(sample5_in)
print("\n".join(result))
assert result == sample5_out.splitlines()

Added: item 1 (benefit=1, weight=1)
Added: item 2 (benefit=6, weight=2)
Added: item 3 (benefit=18, weight=5)
Added: item 4 (benefit=22, weight=6)
Added: item 5 (benefit=28, weight=7)
Capacity set: 11
0/1 Optimal: 40 | Items: [3, 4] | Weight: 11
Fractional Optimal: 42.67 | Fractions: {5: 1.00, 4: 0.67}
Greedy 0/1 (by ratio): 35 | Items: [1, 2, 5] | Weight: 10 | Gap vs Optimal: 5


### Verification Cell

In [15]:
# Error handling
opt = CargoOptimizer()
for bad in [(0, 5), (5, 0), (-1, 3)]:
    try:
        opt.add_item(*bad)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass
try:
    opt.set_capacity(-1)
    raise AssertionError("expected ValueError")
except ValueError:
    pass
opt.add_item(5, 3)
try:
    opt.solve_01()
    raise AssertionError("expected ValueError")
except ValueError:
    pass

# Zero capacity
opt.set_capacity(0)
assert opt.solve_01() == (0, [], 0)
assert opt.solve_greedy_01() == (0, [], 0)
assert opt.solve_fractional()[0] == 0.0

# Item heavier than the capacity
opt = CargoOptimizer()
opt.add_item(100, 50)
opt.add_item(3, 2)
opt.set_capacity(10)
assert opt.solve_01() == (3, [2], 2)
assert opt.solve_greedy_01() == (3, [2], 2)

# Brute-force check of the DP; fractional bound >= 0/1 >= greedy
random.seed(5)
for _ in range(200):
    opt = CargoOptimizer()
    n = random.randint(0, 8)
    for _ in range(n):
        opt.add_item(random.randint(1, 30), random.randint(1, 15))
    opt.set_capacity(random.randint(0, 40))
    best_bf = 0
    for mask in range(1 << n):
        w = sum(opt.items[i][1] for i in range(n) if mask >> i & 1)
        if w <= opt.capacity:
            best_bf = max(best_bf, sum(opt.items[i][0]
                                       for i in range(n) if mask >> i & 1))
    best, ids, wt = opt.solve_01()
    assert best == best_bf
    assert wt <= opt.capacity
    assert best == sum(opt.items[k - 1][0] for k in ids)
    assert opt.solve_fractional()[0] + 1e-9 >= best >= opt.solve_greedy_01()[0]
print("Problem 05: all verification tests passed.")

Problem 05: all verification tests passed.


## Empirical Timing (Problems 01–02)

In [16]:
import time

def time_it(f, arr: list[int]) -> float:
    start = time.perf_counter()
    f(arr)
    return time.perf_counter() - start

algorithms = {
    "Insertion": insertion_sort,
    "Selection": selection_sort,
    "Bubble": bubble_sort,
    "Quick": quick_sort,
    "Merge": merge_sort,
}
sizes = [100, 200, 400, 800]
random.seed(6)
inputs = {
    "sorted": lambda n: list(range(n)),
    "reversed": lambda n: list(range(n, 0, -1)),
    "random": lambda n: random.sample(range(10 * n), n),
}

print(f"{'Algorithm':<10} {'Input':<9}" + "".join(f"{n:>11}" for n in sizes))
for name, f in algorithms.items():
    for kind, gen in inputs.items():
        row = [time_it(f, gen(n)) * 1000 for n in sizes]
        print(f"{name:<10} {kind:<9}" + "".join(f"{t:>9.2f}ms" for t in row))

Algorithm  Input            100        200        400        800
Insertion  sorted        0.02ms     0.04ms     0.11ms     0.17ms
Insertion  reversed      0.99ms     3.99ms    16.88ms    69.45ms
Insertion  random        0.39ms     1.47ms     5.95ms    30.39ms
Selection  sorted        0.50ms     1.55ms     8.99ms    27.44ms
Selection  reversed      0.34ms     1.54ms     6.27ms    25.98ms
Selection  random        0.36ms     1.29ms     5.81ms    22.81ms
Bubble     sorted        0.01ms     0.02ms     0.04ms     0.07ms
Bubble     reversed      0.88ms     3.49ms    14.61ms    64.03ms
Bubble     random        0.69ms     3.12ms    15.59ms    68.43ms
Quick      sorted        0.98ms     3.46ms    15.20ms    68.02ms
Quick      reversed      0.65ms     2.50ms    12.35ms    49.09ms
Quick      random        0.13ms     0.27ms     0.60ms     1.96ms
Merge      sorted        0.24ms     0.50ms     0.83ms     2.15ms
Merge      reversed      0.18ms     0.40ms     1.02ms     2.88ms
Merge      random        

**Comment.** Each doubling of $n$ roughly quadruples the time of Selection Sort on every input and of Insertion and Bubble Sort on reversed and random input, matching their $\Theta(n^2)$ bounds; on sorted input Insertion and Bubble Sort (with the early-exit flag) grow only linearly, showing their adaptive $\Theta(n)$ best case, while Selection Sort is unaffected because it is not adaptive. Merge Sort grows slightly faster than linearly on all three inputs, consistent with $\Theta(n\log n)$ and its lack of a bad input. Quicksort with a last-element pivot is $\Theta(n\log n)$ on random input but quadratic on sorted and reversed input, where every partition is maximally unbalanced, so its curve there behaves like the quadratic sorts.